# Phân Tích Thống Kê Dataset (statistic)

Phân tích chi tiết thống kê về dataset bao gồm: số lượng papers, surveys, duplicate detection, coverage analysis, v.v.

In [1]:
%pip install tabulate

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import glob
from collections import Counter, defaultdict
from tabulate import tabulate

# PATH CONFIG
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(BASE_DIR):
    BASE_DIR = os.path.dirname(os.getcwd())

INPUT_DIR = os.path.join(BASE_DIR, "data_processed")
print(f"📂 Base directory: {BASE_DIR}")
print(f"📂 Input directory: {INPUT_DIR}")

📂 Base directory: e:\Download\NLP DN\NLP DN\data
📂 Input directory: e:\Download\NLP DN\NLP DN\data\data_processed


In [3]:
# UTILITY FUNCTIONS

def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower().strip()
    return re.sub(r'[^a-z0-9]', '', text)


def extract_year(date_str):
    if not date_str:
        return "Unknown"
    try:
        date_str = str(date_str)
        if "-" in date_str:
            return date_str.split("-")[0]
        return date_str
    except:
        return "Unknown"


def is_vietnamese_related(doc):
    keywords = [
        r'\bvietnamese\b',
        r'\bvietnam\b',
        r'\btieng viet\b',
        r'\btiếng việt\b',
        r'\bphobert\b',
        r'\bvi-bert\b',
        r'\bvndt\b',
        r'\bvlsp\b'
    ]
    text = (doc.get("title", "") + " " + doc.get("abstract", "")).lower()
    for pattern in keywords:
        if re.search(pattern, text):
            return True
    return False

print("✅ Utility functions loaded")

✅ Utility functions loaded


## 1. Quét và xử lý các file JSONL

In [4]:
# SCAN FILES
file_pattern = os.path.join(INPUT_DIR, "*.jsonl")
files_to_process = glob.glob(file_pattern)

print(f"🔍 Scanning directory: {INPUT_DIR}")

if not files_to_process:
    print("❌ No JSONL files found.")
else:
    print(f"📁 Found {len(files_to_process)} file(s):")
    for f in files_to_process:
        print(f"   - {os.path.basename(f)}")

🔍 Scanning directory: e:\Download\NLP DN\NLP DN\data\data_processed
📁 Found 1 file(s):
   - final_cleaned_data.jsonl


In [5]:
# INITIALIZE STATISTICS
stats = {
    "total_papers": 0,
    "surveys": 0,
    "papers": 0,
    "duplicates_id": 0,
    "duplicates_title": 0,
    "vietnamese_related": 0,
    "has_citations_list": 0,
    "has_references_list": 0,
    "missing": {
        "abstract": 0,
        "authors": 0,
        "venue": 0,
        "publication_date": 0,
        "citations_list": 0,
        "references_list": 0,
        "pdf_link": 0
    },
    "years": Counter(),
    "venues": Counter(),
    "authors_count": Counter(),
    "abstract_word_counts": [],
    "citations_lengths": [],
    "references_lengths": [],
    "sources": {
        "arxiv": 0,
        "acl": 0,
        "doi": 0
    }
}

# Auxiliary structures
seen_ids = set()
title_map = defaultdict(list)
vietnamese_ids = []

print("✅ Statistics structures initialized")

✅ Statistics structures initialized


In [6]:
# PROCESS ALL FILES
for file_path in files_to_process:
    file_name = os.path.basename(file_path)
    print(f"\n⏳ Processing: {file_name}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                line = line.strip()
                if not line:
                    continue
                
                doc = json.loads(line)
                stats["total_papers"] += 1
                
                # DUPLICATE ID CHECK
                p_id = str(doc.get("paper_id"))
                if p_id in seen_ids:
                    stats["duplicates_id"] += 1
                else:
                    seen_ids.add(p_id)
                
                # DUPLICATE TITLE CHECK
                title = doc.get("title", "")
                norm_title = normalize_text(title)
                if norm_title:
                    title_map[norm_title].append({
                        "id": p_id,
                        "title": title,
                        "file_source": file_name
                    })
                
                # SURVEY / PAPER
                if doc.get("is_survey"):
                    stats["surveys"] += 1
                else:
                    stats["papers"] += 1
                
                # VIETNAMESE RELATED
                if is_vietnamese_related(doc):
                    stats["vietnamese_related"] += 1
                    vietnamese_ids.append(p_id)
                
                # YEAR
                year = extract_year(doc.get("publication_date") or doc.get("year"))
                stats["years"][year] += 1
                
                # VENUE
                venue = doc.get("venue")
                if venue:
                    stats["venues"][venue] += 1
                else:
                    stats["venues"]["Unknown"] += 1
                    stats["missing"]["venue"] += 1
                
                # ABSTRACT
                abstract = doc.get("abstract")
                if not abstract:
                    stats["missing"]["abstract"] += 1
                else:
                    stats["abstract_word_counts"].append(len(abstract.split()))
                
                # AUTHORS
                authors = doc.get("authors", [])
                if not authors:
                    stats["missing"]["authors"] += 1
                else:
                    for author in authors:
                        if isinstance(author, dict) and author.get("name"):
                            stats["authors_count"][author["name"]] += 1
                
                # PUBLICATION DATE
                if not doc.get("publication_date"):
                    stats["missing"]["publication_date"] += 1
                
                # NETWORK DATA
                network = doc.get("network", {})
                citations = network.get("citations", [])
                references = network.get("references", [])
                
                if citations:
                    stats["has_citations_list"] += 1
                else:
                    stats["missing"]["citations_list"] += 1
                
                if references:
                    stats["has_references_list"] += 1
                else:
                    stats["missing"]["references_list"] += 1
                
                stats["citations_lengths"].append(len(citations))
                stats["references_lengths"].append(len(references))
                
                # EXTERNAL LINKS
                ext = doc.get("externalsid", {})
                has_link = False
                
                if ext.get("arxiv"):
                    stats["sources"]["arxiv"] += 1
                    has_link = True
                
                if ext.get("acl"):
                    stats["sources"]["acl"] += 1
                    has_link = True
                
                if ext.get("doi"):
                    stats["sources"]["doi"] += 1
                    has_link = True
                
                if not has_link and not ext.get("s2_url"):
                    stats["missing"]["pdf_link"] += 1
            
            except json.JSONDecodeError:
                continue
    
    print(f"   ✅ Processed {stats['total_papers']:,} papers so far")

print(f"\n✅ Processing complete!")


⏳ Processing: final_cleaned_data.jsonl
   ✅ Processed 13,013 papers so far

✅ Processing complete!


## 2. Phát hiện các bài báo trùng lặp

In [7]:
# DETECT DUPLICATE TITLES
duplicate_groups = {}

for norm_title, papers_list in title_map.items():
    if len(papers_list) > 1:
        duplicate_groups[norm_title] = papers_list
        stats["duplicates_title"] += (len(papers_list) - 1)

print(f"🔍 Duplicate Analysis:")
print(f"   - Duplicate IDs: {stats['duplicates_id']}")
print(f"   - Duplicate Titles: {stats['duplicates_title']}")
print(f"   - Duplicate Title Groups: {len(duplicate_groups)}")

if duplicate_groups:
    print(f"\n   Top duplicates (first 5):")
    for i, (norm_title, papers) in enumerate(list(duplicate_groups.items())[:5], 1):
        print(f"   {i}. {len(papers)} copies: {papers[0]['title'][:60]}...")

🔍 Duplicate Analysis:
   - Duplicate IDs: 0
   - Duplicate Titles: 0
   - Duplicate Title Groups: 0


## 3. Tổng quan Dataset

In [8]:
print("\n" + "="*70)
print("📊 DATASET OVERVIEW")
print("="*70)

total = stats["total_papers"] or 1

overview_data = [
    ["Total papers", f"{stats['total_papers']:,}"],
    ["Papers", f"{stats['papers']:,}"],
    ["Surveys", f"{stats['surveys']:,}"],
    ["Vietnam-related", f"{stats['vietnamese_related']:,}"]
]

for row in overview_data:
    print(f"{row[0]:<25} {row[1]:>15}")


📊 DATASET OVERVIEW
Total papers                       13,013
Papers                             12,771
Surveys                               242
Vietnam-related                        45


## 4. Chất lượng dữ liệu

In [9]:
print("\n⚠️  DATA QUALITY")
print("-"*70)

quality_data = [
    ["Duplicate IDs", f"{stats['duplicates_id']}"],
    ["Duplicate titles", f"{stats['duplicates_title']}"],
    ["Missing abstract", f"{stats['missing']['abstract']} ({stats['missing']['abstract']/total*100:.1f}%)"],
    ["Missing authors", f"{stats['missing']['authors']} ({stats['missing']['authors']/total*100:.1f}%)"],
    ["Missing venue", f"{stats['missing']['venue']} ({stats['missing']['venue']/total*100:.1f}%)"],
    ["Missing publication_date", f"{stats['missing']['publication_date']} ({stats['missing']['publication_date']/total*100:.1f}%)"],
    ["Missing citations list", f"{stats['missing']['citations_list']} ({stats['missing']['citations_list']/total*100:.1f}%)"],
    ["Missing references list", f"{stats['missing']['references_list']} ({stats['missing']['references_list']/total*100:.1f}%)"],
    ["Missing pdf_link", f"{stats['missing']['pdf_link']} ({stats['missing']['pdf_link']/total*100:.1f}%)"]
]

for row in quality_data:
    print(f"{row[0]:<25} {row[1]:>15}")


⚠️  DATA QUALITY
----------------------------------------------------------------------
Duplicate IDs                           0
Duplicate titles                        0
Missing abstract                 0 (0.0%)
Missing authors                 14 (0.1%)
Missing venue                  120 (0.9%)
Missing publication_date     1607 (12.3%)
Missing citations list       5141 (39.5%)
Missing references list      2772 (21.3%)
Missing pdf_link                 0 (0.0%)


## 5. Phủ sóng Network

In [10]:
print("\n🧠 NETWORK COVERAGE")
print("-"*70)

abstract_pct = (total - stats["missing"]["abstract"]) / total * 100
citations_pct = stats["has_citations_list"] / total * 100
references_pct = stats["has_references_list"] / total * 100

network_data = [
    ["Papers with abstract", f"{total - stats['missing']['abstract']:,}", f"{abstract_pct:.1f}%"],
    ["Papers with citations list", f"{stats['has_citations_list']:,}", f"{citations_pct:.1f}%"],
    ["Papers with references list", f"{stats['has_references_list']:,}", f"{references_pct:.1f}%"]
]

for row in network_data:
    print(f"{row[0]:<30} {row[1]:>12} {row[2]:>10}")

if stats["citations_lengths"]:
    avg_citations = sum(stats["citations_lengths"]) / len(stats["citations_lengths"])
    avg_refs = sum(stats["references_lengths"]) / len(stats["references_lengths"])
    
    print(f"\nAverage citations per paper: {avg_citations:.1f}")
    print(f"Average references per paper: {avg_refs:.1f}")


🧠 NETWORK COVERAGE
----------------------------------------------------------------------
Papers with abstract                 13,013     100.0%
Papers with citations list            7,872      60.5%
Papers with references list          10,241      78.7%

Average citations per paper: 8.5
Average references per paper: 14.6


## 6. Thông tin tác giả

In [11]:
print("\n👥 AUTHORS")
print("-"*70)
print(f"Unique authors: {len(stats['authors_count']):,}")

print(f"\nTop 10 most prolific authors:")
for i, (author, count) in enumerate(stats["authors_count"].most_common(10), 1):
    print(f"{i:2d}. {author:<40} {count:>5} papers")


👥 AUTHORS
----------------------------------------------------------------------
Unique authors: 32,321

Top 10 most prolific authors:
 1. Min Zhang                                   72 papers
 2. Iryna Gurevych                              61 papers
 3. Graham Neubig                               58 papers
 4. Yue Zhang                                   50 papers
 5. Yang Liu                                    49 papers
 6. Ryan Cotterell                              48 papers
 7. Dan Roth                                    48 papers
 8. Heng Ji                                     45 papers
 9. Noah A. Smith                               44 papers
10. Qun Liu                                     43 papers


## 7. Phân phối theo năm

In [12]:
print("\n📅 YEAR DISTRIBUTION")
print("-"*70)

sorted_years = sorted(stats["years"].items(), key=lambda x: x[0], reverse=True)

for year, count in sorted_years:
    if year != "Unknown":
        pct = (count / total) * 100
        bar = "█" * int(pct / 2)
        print(f"{year:<10} {count:>6,} papers ({pct:>5.1f}%) {bar}")

# Unknown
unknown_count = stats["years"].get("Unknown", 0)
if unknown_count > 0:
    pct = (unknown_count / total) * 100
    print(f"{'Unknown':<10} {unknown_count:>6,} papers ({pct:>5.1f}%)")


📅 YEAR DISTRIBUTION
----------------------------------------------------------------------
2026           37 papers (  0.3%) 
2025        1,208 papers (  9.3%) ████
2024        1,675 papers ( 12.9%) ██████
2023        1,166 papers (  9.0%) ████
2022          850 papers (  6.5%) ███
2021          839 papers (  6.4%) ███
2020          692 papers (  5.3%) ██
2019          792 papers (  6.1%) ███
2018          630 papers (  4.8%) ██
2017          438 papers (  3.4%) █
2016          432 papers (  3.3%) █
2015          355 papers (  2.7%) █
2014          413 papers (  3.2%) █
2013           38 papers (  0.3%) 
2012           12 papers (  0.1%) 
2011           18 papers (  0.1%) 
2010           16 papers (  0.1%) 
2009          282 papers (  2.2%) █
2008           57 papers (  0.4%) 
2007          122 papers (  0.9%) 
2006          219 papers (  1.7%) 
2005           92 papers (  0.7%) 
2004           87 papers (  0.7%) 
2003          170 papers (  1.3%) 
2002           80 papers (  0.6%) 
2

## 8. Top venues

In [13]:
print("\n🏛️  TOP VENUES")
print("-"*70)
print(f"Unique venues: {len(stats['venues']):,}")

print(f"\nTop 15 venues by paper count:")
for i, (venue, count) in enumerate(stats["venues"].most_common(15), 1):
    pct = (count / total) * 100
    print(f"{i:2d}. {venue:<40} {count:>5} papers ({pct:>5.1f}%)")


🏛️  TOP VENUES
----------------------------------------------------------------------
Unique venues: 1,547

Top 15 venues by paper count:
 1. Annual Meeting of the Association for Computational Linguistics  5136 papers ( 39.5%)
 2. North American Chapter of the Association for Computational Linguistics  1768 papers ( 13.6%)
 3. Conference of the European Chapter of the Association for Computational Linguistics   718 papers (  5.5%)
 4. arXiv.org                                  372 papers (  2.9%)
 5. Transactions of the Association for Computational Linguistics   308 papers (  2.4%)
 6. AAAI Conference on Artificial Intelligence   289 papers (  2.2%)
 7. NAACL-HLT                                  177 papers (  1.4%)
 8. Conference on Empirical Methods in Natural Language Processing   176 papers (  1.4%)
 9. Unknown                                    120 papers (  0.9%)
10. IEEE Access                                 98 papers (  0.8%)
11. WMT@ACL                                     4

## 9. Phân tích Abstract

In [14]:
if stats["abstract_word_counts"]:
    print("\n📝 ABSTRACT ANALYSIS")
    print("-"*70)
    
    word_counts = stats["abstract_word_counts"]
    total_words = sum(word_counts)
    avg_words = total_words / len(word_counts)
    min_words = min(word_counts)
    max_words = max(word_counts)
    
    print(f"Papers with abstract: {len(word_counts):,}")
    print(f"Total words in abstracts: {total_words:,}")
    print(f"Average words per abstract: {avg_words:.1f}")
    print(f"Min words: {min_words}")
    print(f"Max words: {max_words}")


📝 ABSTRACT ANALYSIS
----------------------------------------------------------------------
Papers with abstract: 13,013
Total words in abstracts: 1,871,649
Average words per abstract: 143.8
Min words: 1
Max words: 2621


## 10. Nguồn dữ liệu

In [15]:
print("\n🔗 EXTERNAL SOURCES")
print("-"*70)

sources_data = [
    ["ArXiv", f"{stats['sources']['arxiv']:,}", f"{stats['sources']['arxiv']/total*100:.1f}%"],
    ["ACL", f"{stats['sources']['acl']:,}", f"{stats['sources']['acl']/total*100:.1f}%"],
    ["DOI", f"{stats['sources']['doi']:,}", f"{stats['sources']['doi']/total*100:.1f}%"]
]

for row in sources_data:
    print(f"{row[0]:<25} {row[1]:>15} {row[2]:>10}")


🔗 EXTERNAL SOURCES
----------------------------------------------------------------------
ArXiv                               6,025      46.3%
ACL                                 7,213      55.4%
DOI                                12,727      97.8%


## 11. Danh sách các bài báo liên quan Việt Nam

In [16]:
print(f"\n🇻🇳 VIETNAMESE-RELATED PAPERS")
print("-"*70)
print(f"Total: {stats['vietnamese_related']} papers")
print(f"\nFirst 20 Paper IDs:")
for i, pid in enumerate(vietnamese_ids[:20], 1):
    print(f"{i:2d}. {pid}")

if len(vietnamese_ids) > 20:
    print(f"... and {len(vietnamese_ids) - 20} more papers")


🇻🇳 VIETNAMESE-RELATED PAPERS
----------------------------------------------------------------------
Total: 45 papers

First 20 Paper IDs:
 1. 245634480
 2. 201103583
 3. 276389900
 4. 219324331
 5. 284629544
 6. 221364878
 7. 264396469
 8. 269091208
 9. 276966741
10. 275733863
11. 32228701
12. 255043404
13. 12627600
14. 3113377
15. 11324829
16. 15591672
17. 268681350
18. 253370581
19. 51875779
20. 18250584
... and 25 more papers


## 12. Danh sách Missing Data chi tiết

In [17]:
print("\n📋 MISSING DATA DETAILED BREAKDOWN")
print("="*70)

missing_data_detailed = [
    ["Abstract", f"{stats['missing']['abstract']:,}", f"{stats['missing']['abstract']/total*100:.1f}%"],
    ["Authors", f"{stats['missing']['authors']:,}", f"{stats['missing']['authors']/total*100:.1f}%"],
    ["Venue", f"{stats['missing']['venue']:,}", f"{stats['missing']['venue']/total*100:.1f}%"],
    ["Publication Date", f"{stats['missing']['publication_date']:,}", f"{stats['missing']['publication_date']/total*100:.1f}%"],
    ["Citations List", f"{stats['missing']['citations_list']:,}", f"{stats['missing']['citations_list']/total*100:.1f}%"],
    ["References List", f"{stats['missing']['references_list']:,}", f"{stats['missing']['references_list']/total*100:.1f}%"],
    ["PDF Link", f"{stats['missing']['pdf_link']:,}", f"{stats['missing']['pdf_link']/total*100:.1f}%"]
]

print(f"\nTotal Papers: {total:,}\n")
for row in missing_data_detailed:
    print(f"{row[0]:<20} {row[1]:>15} {row[2]:>12}")


📋 MISSING DATA DETAILED BREAKDOWN

Total Papers: 13,013

Abstract                           0         0.0%
Authors                           14         0.1%
Venue                            120         0.9%
Publication Date               1,607        12.3%
Citations List                 5,141        39.5%
References List                2,772        21.3%
PDF Link                           0         0.0%


## 14. Report Structure (JSON Format)

In [18]:
import json

# Build complete report 
report = {
    "summary": {
        "total_files": len(files_to_process),
        "total_papers": stats["total_papers"],
        "papers": stats["papers"],
        "surveys": stats["surveys"],
        "vietnamese_related": stats["vietnamese_related"],
        "duplicates_id": stats["duplicates_id"],
        "duplicates_title": stats["duplicates_title"]
    },
    "missing_data": stats["missing"],
    "distributions": {
        "years": dict(stats["years"]),
        "venues": dict(stats["venues"])
    },
    "statistics": {
        "abstract_avg_len": (
            sum(stats["abstract_word_counts"]) / len(stats["abstract_word_counts"])
            if stats["abstract_word_counts"] else 0
        ),
        "citations_avg": (
            sum(stats["citations_lengths"]) / len(stats["citations_lengths"])
            if stats["citations_lengths"] else 0
        ),
        "references_avg": (
            sum(stats["references_lengths"]) / len(stats["references_lengths"])
            if stats["references_lengths"] else 0
        )
    },
    "sources": stats["sources"],
    "details": {
        "vietnamese_ids_count": len(vietnamese_ids),
        "duplicate_title_groups": len(duplicate_groups)
    }
}

# Save report to JSON file
REPORT_FILE = os.path.join(INPUT_DIR, "report.json")

with open(REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

print(f"\n✅ Report saved to: {REPORT_FILE}")
print(f"📦 File size: {os.path.getsize(REPORT_FILE) / 1024:.2f} KB")


✅ Report saved to: e:\Download\NLP DN\NLP DN\data\data_processed\report.json
📦 File size: 105.32 KB


## 14. Tóm tắt kết quả

In [19]:
print("\n" + "="*70)
print("✅ SUMMARY REPORT")
print("="*70)

summary = {
    "total_files": len(files_to_process),
    "total_papers": stats["total_papers"],
    "papers": stats["papers"],
    "surveys": stats["surveys"],
    "vietnamese_related": stats["vietnamese_related"],
    "duplicates_id": stats["duplicates_id"],
    "duplicates_title": stats["duplicates_title"],
    "abstract_coverage_percent": round(abstract_pct, 2),
    "citations_coverage_percent": round(citations_pct, 2),
    "references_coverage_percent": round(references_pct, 2),
    "unique_authors": len(stats['authors_count']),
    "unique_venues": len(stats['venues'])
}

for key, value in summary.items():
    key_display = key.replace('_', ' ').title()
    print(f"{key_display:<35} {value:>20,}" if isinstance(value, int) else f"{key_display:<35} {value:>20}")

print("="*70)
print(f"\n✅ Analysis completed successfully!")


✅ SUMMARY REPORT
Total Files                                            1
Total Papers                                      13,013
Papers                                            12,771
Surveys                                              242
Vietnamese Related                                    45
Duplicates Id                                          0
Duplicates Title                                       0
Abstract Coverage Percent                          100.0
Citations Coverage Percent                         60.49
References Coverage Percent                         78.7
Unique Authors                                    32,321
Unique Venues                                      1,547

✅ Analysis completed successfully!
